# Лабораторная работа №12

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

### 1. Загрузите данные из файла abalone.csv. 
##### Это датасет, в котором требуется предсказать возраст ракушки (число колец) по физическим измерениям.

In [2]:
data = pd.read_csv('abalone.csv')

### 2. Преобразуйте признак Sex в числовой: значение F должно перейти в -1, I — в 0, M — в 1. 
##### Если вы используете Pandas, то подойдет следующий код: data[’Sex’] = data[’Sex’].map(lambda x: 1 if x == ’M’ else (-1 if x == ’F’ else 0))


In [3]:
data['Sex'] = data['Sex'].map(lambda x: 1 if x == 'M' else (-1 if x == 'F' else 0))

### 3. Разделите содержимое файлов на признаки и целевую переменную.
##### В последнем столбце записана целевая переменная, в остальных — признаки.

In [4]:
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

### 4. Обучите случайный лес (sklearn.ensemble.RandomForestRegressor) с различным числом деревьев: от 1 до 50 (random_state=1). 
##### Для каждого из вариантов оцените качество работы полученного леса на кросс-валидации по 5 блокам. Используйте параметры "random_state=1"и "shuffle=True"при создании генератора кроссвалидации sklearn.cross_validation.KFold. В качестве меры качества воспользуйтесь коэффициентом детерминации (sklearn.metrics.r2_score).

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)

# Словарь для хранения среднего качества при каждом количестве деревьев
results = {}

for n in range(1, 51):
    # Создаем модель случайного леса с random_state=1
    rf = RandomForestRegressor(n_estimators=n, random_state=1)
    
    # Список для хранения качества на каждом фолде
    scores = []
    
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Обучаем модель
        rf.fit(X_train, y_train)
        
        # Предсказываем
        y_pred = rf.predict(X_test)
        
        # Оцениваем качество (R²)
        scores.append(r2_score(y_test, y_pred))
    
    # Сохраняем среднее качество
    results[n] = np.mean(scores)

### 5. Определите, при каком минимальном количестве деревьев случайный лес показывает качество на кросс-валидации выше 0.52. 
##### Это количество и будет ответом на задание.

In [6]:
min_trees = None
for n, score in results.items():
    if score > 0.52:
        min_trees = n
        break

print(min_trees)

21


### 6. Обратите внимание на изменение качества по мере роста числа деревьев. Ухудшается ли оно?

In [7]:
for n in [1, 5, 10, 20, 30, 40, 50]:
    print(f"{n} деревьев: R² = {results[n]:.4f}")

print('Качество не ухудшается')

1 деревьев: R² = 0.1097
5 деревьев: R² = 0.4650
10 деревьев: R² = 0.4954
20 деревьев: R² = 0.5195
30 деревьев: R² = 0.5271
40 деревьев: R² = 0.5295
50 деревьев: R² = 0.5310
Качество не ухудшается
